# 5. Audio

Read Audio and Frames at the same time, match them by time, and get the sound ready for a speech tool such as Whisper.

Log in first with [1. Quick start](01-quick-start.ipynb).

In [ ]:
import threading
import time

import numpy as np

import googlenestcam as gnc

cam = gnc.list_cameras()[0]

## Read both for 10 seconds

`stream.audio()` and `stream.frames()` each run in their own thread. Leaving the `with` block ends both. Each Audio chunk has `.samples` (NumPy int16, samples x channels), `.sample_rate` (48,000) and `.time`, on the same clock as `frame.time`.

In [ ]:
chunks = []
frames = []

with cam.stream(size=480) as stream:
    threading.Thread(target=lambda: chunks.extend(stream.audio()), daemon=True).start()
    threading.Thread(target=lambda: frames.extend(stream.frames()), daemon=True).start()
    time.sleep(10)

print(len(chunks), "Audio chunks,", len(frames), "Frames")

## Match a Frame to its sound

Find the Audio chunk closest in time to a Frame.

In [ ]:
frame = frames[-1]
chunk = min(chunks, key=lambda c: abs(c.time - frame.time))
print(frame.time, chunk.time, chunk.samples.shape)

## Get the sound ready for Whisper

Whisper wants one channel of 16,000 samples a second, as floats from -1 to 1. This joins the chunks, mixes to one channel and keeps every third sample (48,000 / 3 = 16,000). For better sound, resample with a tool such as `scipy.signal.resample_poly`.

In [ ]:
samples = np.concatenate([c.samples for c in chunks])
mono = samples.mean(axis=1) / 32768
speech = mono[::3].astype(np.float32)
print(len(speech) / 16_000, "seconds")

Whisper is not in any extra. Install it yourself with `pip install openai-whisper`; this cell is skipped if it is missing.

In [ ]:
from importlib.util import find_spec

if find_spec("whisper") is None:
    print("Whisper is not installed; skipping.")
else:
    import whisper

    model = whisper.load_model("base")
    print(model.transcribe(speech)["text"])